In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

## <로버스트 회귀>

In [ ]:
print("""
일반적으로 선형 회귀 모델에서는 회귀계수를 추정할 때 잔차의 제곱합을 이용하는 최소 제곱법을 사용한다.
그런데 이런 경우 데이터의 이상치에 의해 전체 추정치가 왜곡되는 문제가 발생한다.
로버스트 회귀는 이런 문제의 대안으로서 잔차 제곱 대신 절댓값의 합이 최소가 되도록 계수를 추정함을서써 이상치의 영향력을 줄인다.
""")

In [7]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data["data"],
                 columns=data["feature_names"]
                 )
y = pd.Series(data["target"], name="target")

y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [9]:
data= pd.concat([X, y], axis=1)
data

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019908,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068330,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005671,-0.045599,-0.034194,-0.032356,-0.002592,0.002864,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022692,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031991,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018118,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017282,-0.037344,-0.013840,-0.024993,-0.011080,-0.046879,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044528,-0.025930,220.0


In [18]:
train = data.iloc[:int(len(data)*0.8), :]
test = data.iloc[int(len(data)*0.8):, :] 

In [25]:
### 로버스트 회귀
from statsmodels.api import RLM

formula = "target ~ age + sex + bmi + bp"

model = RLM.from_formula(formula, data=train).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                    Robust linear Model Regression Results                    
==============================================================================
Dep. Variable:                 target   No. Observations:                  353
Model:                            RLM   Df Residuals:                      348
Method:                          IRLS   Df Model:                            4
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Sat, 23 Aug 2025                                         
Time:                        21:52:24                                         
No. Iterations:                    12                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    151.1304      3.436     43.987      0.000     144.396     157.864
age           21.2456     75.478      0.281      0.778    -126.689     169.180
sex         -139.0880     74.446     -1.868      0.062    -284.999       6.823
bmi          829.2713     81.146     10.220      0.000     670.229     988.314
bp           390.2851     82.789      4.714      0.000     228.021     552.549
==============================================================================

If the model instance has been used for another fit with different fit parameters, then the fit options might not be the correct ones anymore .
"""

In [26]:
### 예측

pred = model.predict(test)
pred

353     95.651472
354    189.870752
355    140.519164
356    110.228620
357    191.330491
          ...    
437    184.589455
438    104.373883
439    138.521789
440    189.240812
441     64.036932
Length: 89, dtype: float64

In [27]:
### 평가
from sklearn.metrics import r2_score

r2 = r2_score(test["target"], pred)
r2

0.4407220688284196

In [ ]:
### sklearn 방식

In [29]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data["data"],
                 columns=data["feature_names"]
                 )
y = pd.Series(data["target"], name="target")
y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [30]:
### 데이터 분할
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(X, y,
                                                                      test_size=0.2,
                                                                      random_state=42
                                                                      )
print(train_input.shape, test_input.shape, train_target.shape, test_target.shape)

(353, 10) (89, 10) (353,) (89,)


In [33]:
### 로버스트 회귀
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import r2_score

hr = HuberRegressor(epsilon=1)
hr.fit(train_input, train_target)
hr_pred = hr.predict(test_input)

## 성능평가
hr_r2 = r2_score(test_target, hr_pred)
hr_r2

0.43717578714665384

## <분위수 회귀>

In [37]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data["data"],
                 columns=data["feature_names"]
                 )
y = pd.Series(data["target"], name="target")

y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [38]:
data = pd.concat([X, y], axis=1)
data

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019908,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068330,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005671,-0.045599,-0.034194,-0.032356,-0.002592,0.002864,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022692,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031991,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018118,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017282,-0.037344,-0.013840,-0.024993,-0.011080,-0.046879,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044528,-0.025930,220.0


In [39]:
train = data.iloc[:int(len(data)*0.8),:]
test = data.iloc[int(len(data)*0.8):,:]

In [50]:
from statsmodels.api import QuantReg

formula = "target ~ age + sex + bmi + bp"

model = QuantReg.from_formula(formula, data=train).fit(q=0.5)   # 중위수 회귀

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                         QuantReg Regression Results                          
==============================================================================
Dep. Variable:                 target   Pseudo R-squared:               0.2354
Model:                       QuantReg   Bandwidth:                       44.21
Method:                 Least Squares   Sparsity:                        167.4
Date:                Sat, 23 Aug 2025   No. Observations:                  353
Time:                        22:47:47   Df Residuals:                      348
                                        Df Model:                            4
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    149.0769      4.457     33.447      0.000     140.311     157.843
age          -43.0420     97.914     -0.440      0.661    -235.620     149.536
sex         -124.8829     96.575     -1.293      0.197    -314.827      65.061
bmi          846.1914    105.266      8.039      0.000     639.153    1053.229
bp           428.7657    107.398      3.992      0.000     217.534     639.997
==============================================================================
"""

In [51]:
## 예측
pred = model.predict(test)
pred

353     93.999693
354    191.671274
355    144.394030
356    108.072907
357    195.406748
          ...    
437    183.206126
438    100.522701
439    134.902684
440    190.184126
441     59.904023
Length: 89, dtype: float64

In [52]:
## 평가
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(test["target"], pred)
mae

48.95336063915839

In [53]:
model.params

Intercept    149.076856
age          -43.042020
sex         -124.882943
bmi          846.191379
bp           428.765652
dtype: float64